# Thai Sign Language — LSTM Classifier Training

> Trains a stacked LSTM model on the preprocessed keypoint arrays produced by `02_preprocessing.ipynb`.

**Assumes** `demo_data/processed/` contains:
```
X_train.npy  X_val.npy  y_train.npy  y_val.npy  labels.json
```

| Cell | Purpose |
|------|---------|
| 1 | Imports & config |
| 2 | Load data |
| 3 | Build model |
| 4 | Callbacks |
| 5 | Train |
| 6 | Training curves |
| 7 | Evaluation on val set |
| 8 | Per-class analysis |
| 9 | Save model info |
| 10 | Quick inference test |

---
## Cell 1 — Imports & Configuration

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback
)

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# ── Configuration ─────────────────────────────────────────────────────────────
PROCESSED_DIR   = '../demo_data/processed'
MODEL_SAVE_PATH = '../models/lstm_model.keras'

# 50 augments × 50 train files ÷ 32 batch = 78 steps/epoch
AUGMENTS_PER_VIDEO = 50
BATCH_SIZE         = 32

LSTM_UNITS      = [64, 32]   # smaller → harder to memorize 50 source sequences
DROPOUT_RATE    = 0.65       # was 0.55
L2_REG          = 2e-2       # was 5e-3 — much stronger weight decay
LABEL_SMOOTHING = 0.15       # was 0.1
MIXUP_ALPHA     = 0.3        # MixUp Beta(alpha,alpha) — 0=no mix, higher=more mixing

EPOCHS        = 200
LEARNING_RATE = 1e-4         # was 5e-4 — slower → regularization has more time to work

Path('../models').mkdir(parents=True, exist_ok=True)

print(f'TensorFlow         : {tf.__version__}')
print(f'AUGMENTS_PER_VIDEO : {AUGMENTS_PER_VIDEO}')
print(f'LSTM_UNITS         : {LSTM_UNITS}  (smaller to reduce memorization)')
print(f'DROPOUT_RATE       : {DROPOUT_RATE}')
print(f'L2_REG             : {L2_REG}')
print(f'LABEL_SMOOTHING    : {LABEL_SMOOTHING}')
print(f'MIXUP_ALPHA        : {MIXUP_ALPHA}  (MixUp — prevents memorization)')
print(f'LEARNING_RATE      : {LEARNING_RATE}  (lower → slower fitting)')
print(f'EPOCHS             : {EPOCHS}')

TensorFlow         : 2.17.0
AUGMENTS_PER_VIDEO : 50
LSTM_UNITS         : [64, 32]  (smaller to reduce memorization)
DROPOUT_RATE       : 0.65
L2_REG             : 0.02
LABEL_SMOOTHING    : 0.15
MIXUP_ALPHA        : 0.3  (MixUp — prevents memorization)
LEARNING_RATE      : 0.0001  (lower → slower fitting)
EPOCHS             : 200


---
## Cell 2 — Load Data

In [ ]:
processed_dir = Path(PROCESSED_DIR)

# ── Labels + val / test (no augmentation — raw sequences) ─────────────────────
with open(processed_dir / 'labels.json', encoding='utf-8') as f:
    labels_info = json.load(f)
class_names  = labels_info['classes']
class_to_idx = labels_info['class_to_idx']
num_classes  = len(class_names)

X_val  = np.load(processed_dir / 'X_val.npy')   # (25, 60, 404)  base, Person 1
y_val  = np.load(processed_dir / 'y_val.npy')
X_test = np.load(processed_dir / 'X_test.npy')  # (25, 60, 404)  same as val
y_test = np.load(processed_dir / 'y_test.npy')

y_val_oh  = keras.utils.to_categorical(y_val,  num_classes=num_classes)
y_test_oh = keras.utils.to_categorical(y_test, num_classes=num_classes)

# ── Train file list ────────────────────────────────────────────────────────────
with open(processed_dir / 'train_files.json', encoding='utf-8') as f:
    train_info = json.load(f)
train_file_paths  = train_info['files']
train_file_labels = train_info['labels']
target_frames     = train_info['target_frames']
n_features        = train_info['n_features']
n_train_files     = len(train_file_paths)
steps_per_epoch   = (AUGMENTS_PER_VIDEO * n_train_files) // BATCH_SIZE

print('─' * 65)
print(f'Train sources  : {n_train_files} files  (_02 + _03 + _04, globally normalised)')
print(f'Val   samples  : {X_val.shape}  (base, Person 1 — truly unseen)')
print(f'steps_per_epoch: {steps_per_epoch}  '
      f'= {AUGMENTS_PER_VIDEO} aug × {n_train_files} files ÷ {BATCH_SIZE} batch')
print('─' * 65)

# ── Realistic augmentation helpers ────────────────────────────────────────────
HAND_DIM = 126
POSE_DIM = 75
D        = HAND_DIM + POSE_DIM   # 201

def _resample_raw(seq, t):
    n = len(seq)
    if n == t: return seq.copy()
    xo = np.linspace(0,1,n); xn = np.linspace(0,1,t)
    return np.stack([np.interp(xn,xo,seq[:,i]) for i in range(seq.shape[1])],1).astype(np.float32)

def _hflip(seq):
    fl = seq.copy()
    for bo in [0, D]:
        fl[:,bo:bo+63]=seq[:,bo+63:bo+126]; fl[:,bo+63:bo+126]=seq[:,bo:bo+63]
        fl[:,list(range(bo,bo+126,3))]*=-1
        xp=list(range(bo+HAND_DIM,bo+HAND_DIM+POSE_DIM,3))
        if xp and xp[-1]<fl.shape[1]: fl[:,xp]*=-1
    return fl

def _rot2d(seq, deg):
    ca,sa=float(np.cos(np.radians(deg))),float(np.sin(np.radians(deg)))
    r=seq.copy()
    for bo in [0,D]:
        for j in range(bo,min(bo+D,seq.shape[1])-1,3):
            x,y=seq[:,j].copy(),seq[:,j+1].copy()
            r[:,j]=x*ca-y*sa; r[:,j+1]=x*sa+y*ca
    return r.astype(np.float32)

def _random_augment(seq):
    """12 realistic augmentation families (subtle variation only)."""
    T = len(seq)
    c = np.random.randint(0, 12)
    if   c == 0:  out = seq.copy()
    elif c == 1:  out = _hflip(seq)
    elif c == 2:  out = _rot2d(seq, np.random.uniform(-3.0,  3.0))
    elif c == 3:  out = _rot2d(seq, np.random.uniform(-5.0,  5.0))
    elif c == 4:  out = (seq + np.random.normal(0, 0.003, seq.shape)).astype(np.float32)
    elif c == 5:  out = (seq + np.random.normal(0, 0.006, seq.shape)).astype(np.float32)
    elif c == 6:
        factor = np.random.uniform(0.90, 0.95); new_len = max(2, int(T * factor))
        s = _resample_raw(seq, new_len)
        out = np.concatenate([s, np.tile(s[-1:], (T - new_len, 1))], axis=0).astype(np.float32)
    elif c == 7:  out = _resample_raw(seq, int(T * np.random.uniform(1.05, 1.10)))[:T]
    elif c == 8:  out = np.roll(seq,  np.random.randint(2, 4), axis=0).astype(np.float32)
    elif c == 9:  out = np.roll(seq, -np.random.randint(2, 4), axis=0).astype(np.float32)
    elif c == 10: out = _hflip(seq) + np.random.normal(0, 0.003, seq.shape).astype(np.float32)
    else:         out = _rot2d(seq, np.random.uniform(-3.0, 3.0)) + np.random.normal(0, 0.003, seq.shape).astype(np.float32)
    return out.astype(np.float32)

def _py_load_augment(fp_t, lbl_t):
    seq = np.load(fp_t.numpy().decode('utf-8')).astype(np.float32)
    return _random_augment(seq), np.int32(lbl_t.numpy())

def tf_map(fp, lbl):
    seq, lbl2 = tf.py_function(_py_load_augment, [fp, lbl], [tf.float32, tf.int32])
    seq.set_shape([target_frames, n_features])
    lbl2.set_shape([])
    return seq, lbl2

# ── MixUp augmentation ────────────────────────────────────────────────────────
def mixup_batch(x, y, alpha=MIXUP_ALPHA):
    batch_size = tf.shape(x)[0]
    g1 = tf.random.gamma([1], alpha=alpha)
    g2 = tf.random.gamma([1], alpha=alpha)
    lam = tf.cast(g1 / (g1 + g2 + 1e-8), tf.float32)[0]
    lam = tf.maximum(lam, 1.0 - lam)
    indices = tf.random.shuffle(tf.range(batch_size))
    x_mix = lam * x + (1.0 - lam) * tf.gather(x, indices)
    y_mix = lam * y + (1.0 - lam) * tf.gather(y, indices)
    return x_mix, y_mix

# ── tf.data training pipeline ─────────────────────────────────────────────────
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_file_paths, train_file_labels))
    .shuffle(n_train_files, seed=42, reshuffle_each_iteration=True)
    .repeat()
    .map(tf_map, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(buffer_size=512, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE, drop_remainder=True)
    .map(lambda x, y: mixup_batch(x, tf.one_hot(y, num_classes)))
    .prefetch(tf.data.AUTOTUNE)
)

print('tf.data pipeline ready (with MixUp).')
print(f'  {steps_per_epoch} steps/epoch × {BATCH_SIZE} batch  '
      f'({AUGMENTS_PER_VIDEO} aug × MixUp α={MIXUP_ALPHA})')

─────────────────────────────────────────────────────────────────
Train sources  : 50 files  (_02 + _04, globally normalised)
Val   samples  : (25, 60, 404)  (_03, NO augmentation)
Test  samples  : (25, 60, 404)  (base, NO augmentation)
steps_per_epoch: 78  = 50 aug × 50 files ÷ 32 batch
─────────────────────────────────────────────────────────────────
tf.data pipeline ready (with MixUp).
  78 steps/epoch × 32 batch = 2496 samples  (50 aug × MixUp α=0.3)


---
## Cell 3 — Build Model

Architecture:
```
Input(target_frames, F)
  → Masking(0.0)          # ignore zero-padded frames
  → LSTM(128, return_sequences=True)
  → Dropout(0.5)
  → LSTM(64)
  → Dropout(0.5)
  → Dense(64, relu)
  → BatchNormalization()
  → Dropout(0.3)
  → Dense(num_classes, softmax)
```

In [ ]:
def build_model(input_shape, num_classes, lstm_units, dropout_rate,
                learning_rate, l2_reg, label_smoothing):
    """
    2-layer LSTM with strong regularization.
    Simpler than before — with only ~100 source training videos,
    fewer parameters + strong L2/dropout generalize better.

    Input(60, 404)
      → Masking
      → LSTM(128, return_seq=True, L2=5e-3)  → Dropout(0.55)
      → LSTM(64,  return_seq=False, L2=5e-3) → Dropout(0.55)
      → Dense(64, relu, L2=5e-3)
      → BatchNorm → Dropout(0.4)
      → Dense(25, softmax)
    """
    reg = regularizers.l2(l2_reg)

    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Masking(mask_value=0.0),

        layers.LSTM(lstm_units[0], return_sequences=True,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        layers.Dropout(dropout_rate),

        layers.LSTM(lstm_units[1], return_sequences=False,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        layers.Dropout(dropout_rate),

        layers.Dense(64, activation='relu', kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(num_classes, activation='softmax'),
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=['accuracy'],
    )
    return model


input_shape = (target_frames, n_features)
model = build_model(input_shape, num_classes, LSTM_UNITS, DROPOUT_RATE,
                    LEARNING_RATE, L2_REG, LABEL_SMOOTHING)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking (Masking)               │ (None, 60, 404)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 60, 64)         │       120,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │           825 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,489 (525.35 KB)

 Trainable params: 134,425 (525.10 KB)

 Non-trainable params: 64 (256.00 B)

---
## Cell 4 — Callbacks

In [4]:
class LearningRateLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr = float(self.model.optimizer.learning_rate)
        print(f'  lr={lr:.2e}', end='')

early_stop = EarlyStopping(
    monitor='val_loss', patience=25, restore_best_weights=True, verbose=1)
reduce_lr  = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
checkpoint = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)

callbacks = [early_stop, reduce_lr, checkpoint, LearningRateLogger()]
print('Callbacks:', [cb.__class__.__name__ for cb in callbacks])

Callbacks: ['EarlyStopping', 'ReduceLROnPlateau', 'ModelCheckpoint', 'LearningRateLogger']


---
## Cell 5 — Train

In [5]:
print(f'steps_per_epoch = {steps_per_epoch}  '
      f'({AUGMENTS_PER_VIDEO} aug × {n_train_files} files ÷ {BATCH_SIZE} batch)')
print(f'val set         : {X_val.shape}  (Person 3 — unseen during training)')
print()

history = model.fit(
    train_ds,
    validation_data=(X_val, y_val_oh),
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks,
    verbose=1,
)

val_acc_history = history.history['val_accuracy']
best_epoch      = int(np.argmax(val_acc_history)) + 1
best_val_acc    = float(np.max(val_acc_history))

print('\n' + '─' * 50)
print(f'Best val_accuracy : {best_val_acc:.4f}  (epoch {best_epoch})')
print('─' * 50)

steps_per_epoch = 78  (50 aug × 50 files ÷ 32 batch)
val set         : (25, 60, 404)  (Person 3 — unseen during training)

Epoch 1/200
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.0474 - loss: 14.3401
Epoch 1: val_accuracy improved from None to 0.00000, saving model to ../models/lstm_model.keras

Epoch 1: finished saving model to ../models/lstm_model.keras
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.0453 - loss: 14.0551 - val_accuracy: 0.0000e+00 - val_loss: 12.8967 - learning_rate: 1.0000e-04
Epoch 2/200
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.0521 - loss: 13.3063
Epoch 2: val_accuracy improved from 0.00000 to 0.04000, saving model to ../models/lstm_model.keras

Epoch 2: finished saving model to ../models/lstm_model.keras
78/78 ━━━━━━━━━━━━━━━━━━━━ 6s 79ms/step - accuracy: 0.0525 - loss: 13.0551 - val_accuracy: 0.0400 - val_loss: 12.1080 - learning_rate: 1.0000e-04
Epoch 3/200
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.0740 - loss: 12.3437


KeyboardInterrupt: 

---
## Cell 6 — Training Curves

In [ ]:
epochs_ran = len(history.history['accuracy'])
epoch_axis = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=14, fontweight='bold')

# ── Left: Accuracy ────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epoch_axis, history.history['accuracy'],     label='Train', color='#4C72B0', linewidth=2)
ax.plot(epoch_axis, history.history['val_accuracy'], label='Val',   color='#DD8452', linewidth=2)
ax.axvline(best_epoch, color='grey', linestyle='--', linewidth=1.2, label=f'Best epoch ({best_epoch})')
ax.set_title('Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.25)

# ── Right: Loss ───────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epoch_axis, history.history['loss'],     label='Train', color='#4C72B0', linewidth=2)
ax.plot(epoch_axis, history.history['val_loss'], label='Val',   color='#DD8452', linewidth=2)
ax.axvline(best_epoch, color='grey', linestyle='--', linewidth=1.2, label=f'Best epoch ({best_epoch})')
ax.set_title('Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.25)

plt.tight_layout()
curves_path = '../models/training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {curves_path}')

---
## Cell 7 — Evaluation on Val Set

In [ ]:
best_model = keras.models.load_model(MODEL_SAVE_PATH)

y_pred_proba = best_model.predict(X_val, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

# ── Classification report ─────────────────────────────────────────────────────
print('Classification Report')
print('─' * 60)
print(classification_report(
    y_val, y_pred,
    target_names=class_names,
    zero_division=0,
))

# ── Confusion matrix ─────────────────────────────────────────────────────────
cm      = confusion_matrix(y_val, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_norm,
    annot=True, fmt='.2f',
    xticklabels=class_names,
    yticklabels=class_names,
    cmap='Blues',
    linewidths=0.4,
    ax=ax,
)
ax.set_title('Normalized Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

cm_path = '../models/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {cm_path}')

---
## Cell 8 — Per-Class Analysis

In [ ]:
from sklearn.metrics import f1_score

f1_per_class = f1_score(y_val, y_pred, average=None, zero_division=0)

# Sort by F1
sorted_idx = np.argsort(f1_per_class)
worst_idx  = sorted_idx[:3]
best_idx   = sorted_idx[-3:][::-1]

print('Top 3 best-performing classes (by F1):')
for rank, idx in enumerate(best_idx, 1):
    print(f'  {rank}. {class_names[idx]:<22}  F1 = {f1_per_class[idx]:.4f}')

print()
print('Top 3 worst-performing classes (by F1):')
for rank, idx in enumerate(worst_idx, 1):
    print(f'  {rank}. {class_names[idx]:<22}  F1 = {f1_per_class[idx]:.4f}')

# Most-confused partner for the single worst class
worst_class_idx  = worst_idx[0]
worst_class_name = class_names[worst_class_idx]
cm_row           = cm[worst_class_idx].copy()
cm_row[worst_class_idx] = 0  # exclude correct predictions

if cm_row.sum() > 0:
    confused_with_idx  = int(np.argmax(cm_row))
    confused_with_name = class_names[confused_with_idx]
    n_confused         = cm_row[confused_with_idx]
    print()
    print(f'Worst class "{worst_class_name}" most often confused with:')
    print(f'  → "{confused_with_name}"  ({n_confused} times)')
else:
    print(f'\nWorst class "{worst_class_name}" had no mis-classifications.')

---
## Cell 9 — Save Model Info

In [ ]:
model_info = {
    'input_shape'   : [target_frames, n_features],
    'num_classes'   : num_classes,
    'class_names'   : class_names,
    'class_to_idx'  : class_to_idx,
    'best_val_accuracy': round(best_val_acc, 6),
    'feature_config': {
        'extract_face'      : False,
        'extract_pose'      : True,
        'extract_hands'     : True,
        'target_frames'     : target_frames,
        'augment_per_video' : 80,
    },
}

model_info_path = '../models/model_info.json'
with open(model_info_path, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print(f'Saved → {model_info_path}')
print()
print(json.dumps(model_info, ensure_ascii=False, indent=2))

---
## Cell 10 — Quick Inference Test

In [ ]:
# Load model and model info fresh (simulates API server startup)
infer_model = keras.models.load_model(MODEL_SAVE_PATH)

with open('../models/model_info.json', encoding='utf-8') as f:
    info = json.load(f)

infer_classes = info['class_names']


def predict_from_npy(npy_path: str) -> tuple:
    """
    Load a single .npy sequence and return (predicted_word, confidence).

    Parameters
    ----------
    npy_path : path to a (target_frames, F) float32 .npy file

    Returns
    -------
    (predicted_word: str, confidence: float)  confidence in [0, 1]
    """
    seq   = np.load(npy_path).astype(np.float32)  # (T, F)
    batch = seq[np.newaxis, ...]                   # (1, T, F)
    proba = infer_model.predict(batch, verbose=0)[0]  # (num_classes,)
    pred_idx    = int(np.argmax(proba))
    confidence  = float(proba[pred_idx])
    return infer_classes[pred_idx], confidence


# ── Test on 5 random val samples ──────────────────────────────────────────────
rng_test   = np.random.default_rng(7)
test_idx   = rng_test.choice(len(X_val), size=5, replace=False)

# Save samples to temp .npy files to exercise the function as intended
tmp_dir = Path('../models/tmp_inference')
tmp_dir.mkdir(exist_ok=True)

print(f'  {"True label":<22}  {"Predicted":<22}  {"Confidence":>10}  {"Correct"}')
print(f'  {"-"*22}  {"-"*22}  {"-"*10}  {"-"*7}')

for i, vi in enumerate(test_idx):
    tmp_path = tmp_dir / f'sample_{i}.npy'
    np.save(tmp_path, X_val[vi])

    true_label            = infer_classes[int(y_val[vi])]
    predicted_word, conf  = predict_from_npy(str(tmp_path))
    correct               = '✓' if predicted_word == true_label else '✗'

    print(f'  {true_label:<22}  {predicted_word:<22}  {conf*100:>9.1f}%  {correct}')

# Clean up temp files
import shutil
shutil.rmtree(tmp_dir)
print('\nDone.')